# 01 - Data Validation

**Objective**: Validate the raw PubMed mHealth dataset, check for missing values, duplicates, and data quality issues. Produce a cleaned candidate dataset.

**Input**: `data/pubmed_mhealth_2015_2025_proposed_final.xls` (CSV-formatted)

**Outputs**:
- `outputs/data_validation_report.csv`
- `outputs/data_validation_summary.json`
- `outputs/cleaning_filter_log.csv`
- `outputs/mhealth_candidate_cleaned.csv`

In [1]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

In [2]:
from src.config_loader import load_config, set_seed, ensure_dirs
from src.data_validation import (
    load_dataset, validate_dataset, print_validation_report,
    clean_dataset, save_validation_outputs,
)

cfg = load_config()
dirs = ensure_dirs()
set_seed()

[CONFIG] Random seed set to 42


## Load Raw Dataset

In [3]:
df_raw = load_dataset()
df_raw.head()

[DATA] Loading dataset from: E:\Major_Project_Implementation\data\pubmed_mhealth_2015_2025_proposed_final.xls
[DATA] File size: 70.7 MB
[DATA] Detected CSV format (despite .xls extension), using read_csv()


[DATA] Successfully loaded with encoding: utf-8-sig
[DATA] Loaded 14,905 records with 11 columns
[DATA] Columns: ['pmid', 'title', 'abstract', 'year', 'journal', 'authors', 'doi', 'mesh_terms', 'keywords', 'publication_types', 'document']


,pmid,title,abstract,year,journal,authors,doi,mesh_terms,keywords,publication_types,document
0,24282189,Developing requirements for a mobile app to su...,Tick bites and tick-borne infections are an in...,2015,Health informatics journal,Lex van Velsen; Desirée J M A Beaujean; Jobke ...,10.1177/1460458213496419,Consumer Health Information; Early Diagnosis; ...,Lyme borreliosis; design; infection prevention...,Journal Article,Developing requirements for a mobile app to su...
1,24313728,"Pilot feasibility study of a brief, tailored m...",This pilot feasibility study investigated a br...,2015,"Behavioral medicine (Washington, D.C.)",Brian K Ahmedani; Nuala Crotty; Muwaffak M Abd...,10.1080/08964289.2013.867827,Adolescent; Adult; Chronic Pain; Cognitive Beh...,brief intervention; chronic pain; depression; ...,Clinical Trial; Journal Article; Research Supp...,"Pilot feasibility study of a brief, tailored m..."
2,24434827,The use of mHealth to deliver tailored message...,BACKGROUND: Evidence supports the role of feed...,2015,The Journal of cardiovascular nursing,Erica J Ambeba; Lei Ye; Susan M Sereika; Mindi...,10.1097/JCN.0000000000000120,"Adolescent; Adult; Computers, Handheld; Dietar...",NaN,Journal Article; Randomized Controlled Trial; ...,The use of mHealth to deliver tailored message...
3,24470587,Implementation of an app-based neuromuscular t...,BACKGROUND/AIM: The contemporary electronic me...,2015,British journal of sports medicine,Ingrid Vriend; Iris Coehoorn; Evert Verhagen,10.1136/bjsports-2013-092896,"Adolescent; Adult; Aged; Aged, 80 and over; An...",Ankle injuries; Implementation; Injury Prevention,Evaluation Study; Journal Article,Implementation of an app-based neuromuscular t...
4,24598552,Family history of stroke among African America...,BACKGROUND: African Americans are at greater r...,2015,The Journal of cardiovascular nursing,Dawn M Aycock; Kenya D Kirkendoll; Kisha C Col...,10.1097/JCN.0000000000000125,Adult; Black or African American; Attitude to ...,NaN,Journal Article,Family history of stroke among African America...


## Run Validation Checks

In [4]:
stats = validate_dataset(df_raw)
print_validation_report(stats)


DATASET VALIDATION REPORT

Total records: 14,905
Columns: ['pmid', 'title', 'abstract', 'year', 'journal', 'authors', 'doi', 'mesh_terms', 'keywords', 'publication_types', 'document']

--- Missing Values ---
  pmid: 0 null, 0 empty
  title: 0 null, 0 empty
  abstract: 0 null, 0 empty
  year: 0 null, 0 empty
  journal: 0 null, 0 empty
  authors: 9 null, 0 empty
  doi: 253 null, 0 empty
  mesh_terms: 5,389 null, 0 empty
  keywords: 1,536 null, 0 empty
  publication_types: 0 null, 0 empty
  document: 0 null, 0 empty

--- Duplicates ---
  duplicate_pmids: 0
  duplicate_dois: 1
  duplicate_documents: 2

--- Year Distribution ---
  2015: 436
  2016: 637
  2017: 798
  2018: 1,009
  2019: 1,234
  2020: 1,570
  2021: 1,707
  2022: 1,910
  2023: 1,718
  2024: 1,717
  2025: 2,169

--- Publication Types (top 20) ---
  Journal Article: 8,701
  Journal Article; Research Support, Non-U.S. Gov't: 2,038
  Journal Article; Randomized Controlled Trial: 493
  Journal Article; Randomized Controlled Trial;

## Clean Dataset

In [5]:
df_cleaned, filter_log = clean_dataset(df_raw)
print(f"\nCleaned dataset shape: {df_cleaned.shape}")
df_cleaned.head()


[CLEAN] Starting dataset cleaning...
  [Remove duplicate PMIDs] before=14,905  removed=0  after=14,905
  [Remove short/missing titles] before=14,905  removed=0  after=14,905
  [Remove short/missing abstracts] before=14,905  removed=23  after=14,882
  [Filter year range 2015-2025] before=14,882  removed=0  after=14,882
  [Remove short documents] before=14,882  removed=0  after=14,882
  [Remove duplicate documents] before=14,882  removed=1  after=14,881

[CLEAN] Final cleaned dataset: 14,881 records

Cleaned dataset shape: (14881, 11)


,pmid,title,abstract,year,journal,authors,doi,mesh_terms,keywords,publication_types,document
0,24282189,Developing requirements for a mobile app to su...,Tick bites and tick-borne infections are an in...,2015,Health informatics journal,Lex van Velsen; Desirée J M A Beaujean; Jobke ...,10.1177/1460458213496419,Consumer Health Information; Early Diagnosis; ...,Lyme borreliosis; design; infection prevention...,Journal Article,Developing requirements for a mobile app to su...
1,24313728,"Pilot feasibility study of a brief, tailored m...",This pilot feasibility study investigated a br...,2015,"Behavioral medicine (Washington, D.C.)",Brian K Ahmedani; Nuala Crotty; Muwaffak M Abd...,10.1080/08964289.2013.867827,Adolescent; Adult; Chronic Pain; Cognitive Beh...,brief intervention; chronic pain; depression; ...,Clinical Trial; Journal Article; Research Supp...,"Pilot feasibility study of a brief, tailored m..."
2,24434827,The use of mHealth to deliver tailored message...,BACKGROUND: Evidence supports the role of feed...,2015,The Journal of cardiovascular nursing,Erica J Ambeba; Lei Ye; Susan M Sereika; Mindi...,10.1097/JCN.0000000000000120,"Adolescent; Adult; Computers, Handheld; Dietar...",NaN,Journal Article; Randomized Controlled Trial; ...,The use of mHealth to deliver tailored message...
3,24470587,Implementation of an app-based neuromuscular t...,BACKGROUND/AIM: The contemporary electronic me...,2015,British journal of sports medicine,Ingrid Vriend; Iris Coehoorn; Evert Verhagen,10.1136/bjsports-2013-092896,"Adolescent; Adult; Aged; Aged, 80 and over; An...",Ankle injuries; Implementation; Injury Prevention,Evaluation Study; Journal Article,Implementation of an app-based neuromuscular t...
4,24598552,Family history of stroke among African America...,BACKGROUND: African Americans are at greater r...,2015,The Journal of cardiovascular nursing,Dawn M Aycock; Kenya D Kirkendoll; Kisha C Col...,10.1097/JCN.0000000000000125,Adult; Black or African American; Attitude to ...,NaN,Journal Article,Family history of stroke among African America...


## Save Outputs

In [6]:
saved = save_validation_outputs(df_raw, stats, filter_log, df_cleaned)
print("\nAll validation outputs saved.")

[SAVE] Validation report -> E:\Major_Project_Implementation\outputs\data_validation_report.csv
[SAVE] Validation summary -> E:\Major_Project_Implementation\outputs\data_validation_summary.json
[SAVE] Cleaning filter log -> E:\Major_Project_Implementation\outputs\cleaning_filter_log.csv


[SAVE] Cleaned dataset -> E:\Major_Project_Implementation\outputs\mhealth_candidate_cleaned.csv  (14,881 records)

All validation outputs saved.
